# LangChain Function Calling Demo

This notebook demonstrates how **LangChain** simplifies the process of using tools with LLMs. 

**Key Improvements over the manual approach:**
1.  **Automatic Schema Generation**: No need to manually write complex JSON schemas. LangChain infers them from your Python function signatures and docstrings.
2.  **Automatic Execution Loop**: The `AgentExecutor` handles the "call LLM -> parse tool -> run tool -> submit result -> get final answer" loop for you.

In [ ]:
# Install necessary libraries
%pip install langchain langchain-openai langchain-community duckdb requests python-dotenv

In [2]:
import os
import json
import requests
import duckdb
from dotenv import load_dotenv

# LangChain Imports
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# Load environment variables
load_dotenv("../.env")

if not os.getenv("OPENAI_API_KEY"):
    print("Warning: OPENAI_API_KEY not found in environment variables.")
else:
    print("API Key loaded successfully.")

API Key loaded successfully.


## 1. Setup Database

Same setup as before: connect to DuckDB and load the CSVs.

In [3]:
conn = duckdb.connect("city_tour.db")

try:
    conn.execute("""
        CREATE OR REPLACE TABLE flight AS 
        SELECT * FROM 'flight_data.csv'
    """)
    
    conn.execute("""
        CREATE OR REPLACE TABLE fun_facts AS 
        SELECT * FROM 'fun_facts.csv'
    """)
    print("Tables loaded successfully.")
except Exception as e:
    print(f"Error loading data: {e}")

Tables loaded successfully.


## 2. Define Tools using `@tool`

This is the **magic part**. By simply adding the `@tool` decorator, LangChain automatically creates the JSON schema required by OpenAI. 

**Important**: The docstring becomes the tool description, and the argument types become the parameter types.

In [4]:
@tool
def get_weather(latitude: float, longitude: float) -> str:
    """Get the current temperature for a given latitude and longitude."""
    # Note: We added type hints (float) which LangChain uses for validation
    print(f"DEBUG: Calling get_weather for {latitude}, {longitude}")
    try:
        url = "https://api.open-meteo.com/v1/forecast"
        params = {
            "latitude": latitude,
            "longitude": longitude,
            "current": "temperature_2m,wind_speed_10m",
            "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m"
        }
        response = requests.get(url, params=params)
        data = response.json()
        return json.dumps(data['current'])
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def get_flight(from_city: str, to_city: str) -> str:
    """Get flight information between two cities."""
    print(f"DEBUG: Calling get_flight from {from_city} to {to_city}")
    try:
        results = conn.execute(
            "SELECT * FROM flight WHERE from_city=? AND to_city=?", 
            [from_city, to_city]
        ).fetchall()
        
        if not results:
            return "No flights found"
        
        columns = [desc[0] for desc in conn.description]
        formatted_results = [
            dict(zip(columns, row)) for row in results
        ]
        return json.dumps(formatted_results)
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def get_fact(city: str) -> str:
    """Get a fun fact about a specific city."""
    print(f"DEBUG: Calling get_fact for {city}")
    try:
        results = conn.execute(
            "SELECT * FROM fun_facts WHERE City=?", 
            [city]
        ).fetchall()
        
        if not results:
            return "No facts found"
        
        columns = [desc[0] for desc in conn.description]
        formatted_results = [
            dict(zip(columns, row)) for row in results
        ]
        return json.dumps(formatted_results)
    except Exception as e:
        return f"Error: {str(e)}"

## 3. Create the Agent

We tie everything together:
1.  **LLM**: `ChatOpenAI`
2.  **Tools**: List of our decorated functions
3.  **Prompt**: A template that includes a place for the agent's "scratchpad" (history of tool calls).

In [5]:
# 1. Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2. create valid list of tools
tools = [get_weather, get_flight, get_fact]

# 3. Create Prompt
# We use a pre-built template or create a simple one
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the available tools to answer the user's questions."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"), # This is where the tool calls/responses go
])

# 4. Create Agent
# create_tool_calling_agent is the modern way to create an agent that uses OpenAI-style tool calling
agent = create_tool_calling_agent(llm, tools, prompt)

# 5. Create Executor
# The executor matches the manual loop we wrote in the other notebook
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

## 4. Run Examples

Now we just call `agent_executor.invoke()` with the user input. The agent handles the rest.

In [6]:
# Example 1: Weather
response = agent_executor.invoke({
    "input": "What is the weather like in these coordinates 25.2048, 55.2708?"
})
print("Assistant:", response['output'])



> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'latitude': 25.2048, 'longitude': 55.2708}`


DEBUG: Calling get_weather for 25.2048, 55.2708
{"time": "2025-12-09T14:00", "interval": 900, "temperature_2m": 26.5, "wind_speed_10m": 11.6}The current weather at the coordinates 25.2048, 55.2708 is as follows:
- Temperature: 26.5°C
- Wind Speed: 11.6 km/h

> Finished chain.
Assistant: The current weather at the coordinates 25.2048, 55.2708 is as follows:
- Temperature: 26.5°C
- Wind Speed: 11.6 km/h


In [7]:
# Example 2: Flights
response = agent_executor.invoke({
    "input": "Is there a flight from Paris to London?"
})
print("Assistant:", response['output'])



> Entering new AgentExecutor chain...

Invoking: `get_flight` with `{'from_city': 'Paris', 'to_city': 'London'}`


DEBUG: Calling get_flight from Paris to London
[{"from_city": "Paris", "to_city": "London", "price": 1188.35, "duration": 868}]Yes, there is a flight from Paris to London. The flight duration is approximately 868 minutes, and the price is around $1188.35.

> Finished chain.
Assistant: Yes, there is a flight from Paris to London. The flight duration is approximately 868 minutes, and the price is around $1188.35.


In [8]:
# Example 3: Complex Query (Flight + Fact)
# Notice how the agent automatically handles calling both tools and synthesizing the answer
response = agent_executor.invoke({
    "input": "Find me a flight from New York to London and tell me a fun fact about London."
})
print("Assistant:", response['output'])



> Entering new AgentExecutor chain...

Invoking: `get_flight` with `{'from_city': 'New York', 'to_city': 'London'}`


DEBUG: Calling get_flight from New York to London
[{"from_city": "New York", "to_city": "London", "price": 1139.03, "duration": 760}, {"from_city": "New York", "to_city": "London", "price": 1314.35, "duration": 261}]
Invoking: `get_fact` with `{'city': 'London'}`


DEBUG: Calling get_fact for London
[{"City": "London", "Fun Fact": "London was the first city to have an underground subway system, known as the Tube."}]I found a couple of flight options from New York to London:

1. **Flight Option 1**: 
   - **Price**: $1139.03
   - **Duration**: 12 hours and 40 minutes

2. **Flight Option 2**: 
   - **Price**: $1314.35
   - **Duration**: 4 hours and 21 minutes

Additionally, here's a fun fact about London: **London was the first city to have an underground subway system, known as the Tube.**

> Finished chain.
Assistant: I found a couple of flight options from New York t